In [ ]:
# =========================================================
# PRACTICA DETECCION DE OBJETOS
# FILTRAR COCO -> PERSON, CHAIR, LAPTOP
# CREAR TRAIN / VAL / TEST
# DATASET GUARDADO EN LOCAL (/content)
# =========================================================

# =========================================================
# 1. MONTAR GOOGLE DRIVE
# (solo para leer el ZIP y las anotaciones originales)
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. IMPORTS
# =========================================================

import os
import json
import shutil
import zipfile

from tqdm import tqdm
from sklearn.model_selection import train_test_split

# =========================================================
# 3. RUTAS DE ORIGEN (Drive — solo lectura)
# =========================================================

DRIVE_BASE = "/content/drive/MyDrive/Datasets"

TRAIN_ZIP_DRIVE = os.path.join(
    DRIVE_BASE,
    "train2017.zip"
)

TRAIN_ANNOTATIONS = os.path.join(
    DRIVE_BASE,
    "annotations",
    "instances_train2017.json"
)

# =========================================================
# 4. RUTAS LOCALES (SSD rápido de Colab)
# =========================================================

LOCAL_ZIP          = "/content/train2017.zip"
LOCAL_TRAIN_FOLDER = "/content/train2017"

# Dataset filtrado: todo en /content (local, persiste mientras
# el runtime esté vivo; usar en la misma sesión)
OUTPUT_BASE = "/content/filtered_dataset"

os.makedirs(OUTPUT_BASE, exist_ok=True)

print(f"Dataset filtrado se guardará en: {OUTPUT_BASE}")

# =========================================================
# 5. COPIAR ZIP A DISCO LOCAL
# =========================================================

if not os.path.exists(LOCAL_ZIP):

    print("\n=================================================")
    print("COPIANDO ZIP A DISCO LOCAL")
    print("=================================================\n")

    total_size = os.path.getsize(TRAIN_ZIP_DRIVE)

    with open(TRAIN_ZIP_DRIVE, 'rb') as src, \
         open(LOCAL_ZIP, 'wb') as dst:

        with tqdm(
            total=total_size,
            unit='B',
            unit_scale=True,
            desc="Copiando ZIP"
        ) as pbar:

            while True:
                chunk = src.read(1024 * 1024)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))

    print("\nZIP copiado correctamente.")

else:
    print("\nZIP local ya existe.")

# =========================================================
# 6. EXTRAER ZIP LOCALMENTE
# =========================================================

if not os.path.exists(LOCAL_TRAIN_FOLDER):

    print("\n=================================================")
    print("EXTRAYENDO TRAIN2017")
    print("=================================================\n")

    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zip_ref:
        members = zip_ref.infolist()
        for member in tqdm(members, desc="Extrayendo archivos"):
            zip_ref.extract(member, "/content")

    print("\nExtracción completada.")

else:
    print("\ntrain2017 ya fue extraido.")

# =========================================================
# 7. CLASES A FILTRAR
# =========================================================

# COCO IDs:
# person = 1
# chair  = 62
# laptop = 73

TARGET_CLASSES = {
    1:  "person",
    62: "chair",
    73: "laptop"
}

# =========================================================
# 8. CARGAR ANOTACIONES COCO
# =========================================================

print("\n=================================================")
print("CARGANDO ANOTACIONES")
print("=================================================\n")

with open(TRAIN_ANNOTATIONS, 'r') as f:
    coco = json.load(f)

print("Anotaciones cargadas correctamente.")

# =========================================================
# 9. FILTRAR CATEGORIAS
# =========================================================

categories = [
    cat for cat in coco['categories']
    if cat['id'] in TARGET_CLASSES
]

print("\nCategorias seleccionadas:\n")
for cat in categories:
    print(cat)

# =========================================================
# 10. FILTRAR ANOTACIONES
# =========================================================

print("\n=================================================")
print("FILTRANDO ANOTACIONES")
print("=================================================\n")

annotations = [
    ann for ann in tqdm(
        coco['annotations'],
        desc="Filtrando anotaciones"
    )
    if ann['category_id'] in TARGET_CLASSES
]

print(f"\nAnotaciones validas: {len(annotations)}")

# =========================================================
# 11. OBTENER IDS VALIDOS
# =========================================================

valid_image_ids = set(
    ann['image_id'] for ann in annotations
)

# =========================================================
# 12. FILTRAR IMAGENES
# =========================================================

images = [
    img for img in tqdm(
        coco['images'],
        desc="Filtrando imagenes"
    )
    if img['id'] in valid_image_ids
]

print(f"\nImagenes validas: {len(images)}")

# =========================================================
# 13. DIVIDIR TRAIN / VAL / TEST
# =========================================================

print("\n=================================================")
print("CREANDO SPLITS")
print("=================================================\n")

image_ids = [img['id'] for img in images]

train_ids, temp_ids = train_test_split(
    image_ids,
    test_size=0.30,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

splits = {
    "train": set(train_ids),
    "val":   set(val_ids),
    "test":  set(test_ids)
}

# =========================================================
# 14. CREAR SPLITS Y GUARDAR EN LOCAL
# =========================================================

for split_name, split_ids in splits.items():

    print("\n=================================================")
    print(f"PROCESANDO {split_name.upper()}")
    print("=================================================\n")

    split_folder = os.path.join(OUTPUT_BASE, split_name)
    os.makedirs(split_folder, exist_ok=True)

    split_images = [
        img for img in images
        if img['id'] in split_ids
    ]

    split_annotations = [
        ann for ann in annotations
        if ann['image_id'] in split_ids
    ]

    print(f"Imagenes: {len(split_images)}")
    print(f"Anotaciones: {len(split_annotations)}")

    # Copiar imágenes a local
    print("\nCopiando imagenes a local...\n")

    for img in tqdm(split_images, desc=f"Copiando {split_name}"):

        src = os.path.join(LOCAL_TRAIN_FOLDER, img['file_name'])
        dst = os.path.join(split_folder, img['file_name'])

        if os.path.exists(src):
            shutil.copy(src, dst)

    # Guardar JSON en local
    json_output = os.path.join(OUTPUT_BASE, f"instances_{split_name}.json")

    with open(json_output, 'w') as f:
        json.dump({
            "images"     : split_images,
            "annotations": split_annotations,
            "categories" : categories
        }, f)

    print(f"\nJSON guardado: {json_output}")

# =========================================================
# 15. RESUMEN FINAL
# =========================================================

print("\n=================================================")
print("DATASET FILTRADO GENERADO")
print("=================================================\n")

print(f"Ruta final: {OUTPUT_BASE}")
print("\nContenido:")
for item in sorted(os.listdir(OUTPUT_BASE)):
    print(f"  {item}")

print("\nClases incluidas:")
print("- person")
print("- chair")
print("- laptop")

print("\nProceso terminado correctamente.")
print("\nNOTA: el dataset está en /content/filtered_dataset")
print("Usa OUTPUT_BASE como DATASET_BASE en los notebooks de entrenamiento.")

Mounted at /content/drive
Dataset filtrado se guardará en: /content/filtered_dataset

COPIANDO ZIP A DISCO LOCAL



Copiando ZIP: 100%|██████████| 19.3G/19.3G [07:13<00:00, 44.6MB/s]



ZIP copiado correctamente.

EXTRAYENDO TRAIN2017



Extrayendo archivos: 100%|██████████| 118288/118288 [03:29<00:00, 564.13it/s]



Extracción completada.

CARGANDO ANOTACIONES

Anotaciones cargadas correctamente.

Categorias seleccionadas:

{'supercategory': 'person', 'id': 1, 'name': 'person'}
{'supercategory': 'furniture', 'id': 62, 'name': 'chair'}
{'supercategory': 'electronic', 'id': 73, 'name': 'laptop'}

FILTRANDO ANOTACIONES



...


JSON guardado: /content/filtered_dataset/instances_train.json

PROCESANDO VAL

Imagenes: 10505
Anotaciones: 45168

Copiando imagenes a local...



Copiando val: 100%|██████████| 10505/10505 [00:34<00:00, 303.97it/s]



JSON guardado: /content/filtered_dataset/instances_val.json

PROCESANDO TEST

Imagenes: 10506
Anotaciones: 45450

Copiando imagenes a local...



Copiando test: 100%|██████████| 10506/10506 [00:34<00:00, 303.63it/s]



JSON guardado: /content/filtered_dataset/instances_test.json

DATASET FILTRADO GENERADO

Ruta final: /content/filtered_dataset

Contenido:
  instances_test.json
  instances_train.json
  instances_val.json
  test
  train
  val

Clases incluidas:
- person
- chair
- laptop

Proceso terminado correctamente.

NOTA: el dataset está en /content/filtered_dataset
Usa OUTPUT_BASE como DATASET_BASE en los notebooks de entrenamiento.


In [ ]:
# =========================================================
# VERIFICACIÓN
# =========================================================

import os

PATH = "/content/filtered_dataset"

print(os.listdir(PATH))

for split in ['train', 'val', 'test']:
    split_path = os.path.join(PATH, split)
    print(f"Imagenes {split}: {len(os.listdir(split_path))}")

['val', 'instances_train.json', 'train', 'instances_test.json', 'test', 'instances_val.json']
Imagenes train: 49025
Imagenes val: 10505
Imagenes test: 10506


In [ ]:
!tar -cf /content/filtered_dataset.tar -C /content filtered_dataset